# Bike Sharing: train, chọn tham số và đánh giá cuối

Notebook độc lập dành cho **day.csv và hour.csv**, mục tiêu hồi quy **cnt**.

1. Tải notebook này lên Google Colab.
2. Tạo thư mục **MyDrive/TT_CSN/data** trên Google Drive, tải hai CSV **gốc** vào đó.
3. Chọn runtime Python 3 (CPU là đủ), chạy các ô từ trên xuống và cho phép kết nối Drive.
4. Mỗi lần chạy lưu vào thư mục mới trong **MyDrive/TT_CSN/runs**.

Chia theo thời gian: **70% train / 15% validation / 15% test**. Làm tròn xuống train và validation,
test nhận phần còn lại. Không dùng test để chọn tham số và không refit trên train + validation.
Hai dataset chạy riêng. `casual` và `registered` bị loại vì tổng của chúng chính là `cnt`.

Random Forest được chọn làm mô hình khởi đầu. Chọn trong 8 cấu hình bằng RMSE validation;
đánh giá cuối bằng RMSE, MAE và R², đối chiếu với dự đoán trung bình train.
Thời tiết phải có sẵn tại thời điểm dự đoán; dự báo tương lai cần đầu vào thời tiết tương ứng.


In [2]:
%pip -q install "scikit-learn>=1.2,<2" "pandas>=1.5,<4" "numpy>=1.24,<3" "joblib>=1.2,<2"


## Thư viện và cấu hình


In [3]:
import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

NUMERIC = ['temp', 'atemp', 'hum', 'windspeed']
BINARY = ['yr', 'holiday', 'workingday']
CATEGORICAL = ['season', 'mnth', 'weekday', 'weathersit']
TARGET = 'cnt'
SEED = 42
DEFAULT_GRID = {
    'n_estimators': [100, 200],
    'max_depth': [None, 16],
    'min_samples_leaf': [1, 3],
    'max_features': [1.0],
}


def save_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False),
                          encoding='utf-8')


## Kết nối Google Drive


In [4]:
def connect_google_drive(project_name='TT_CSN', mount_point='/content/drive'):
    """Yêu cầu đăng nhập Drive trong Colab, trả về thư mục dự án."""
    from google.colab import drive
    drive.mount(mount_point)
    project = Path(mount_point) / 'MyDrive' / project_name
    (project / 'data').mkdir(parents=True, exist_ok=True)
    return project


## Đọc dữ liệu, chia 70/15/15 và xuất CSV gốc


In [5]:
def feature_columns(dataset):
    if dataset not in ('day', 'hour'):
        raise ValueError("dataset phải là 'day' hoặc 'hour'")
    categorical = CATEGORICAL + (['hr'] if dataset == 'hour' else [])
    return NUMERIC + BINARY + categorical, categorical


def normalize_features(frame, features):
    """Giữ đúng schema và xử lý các ký hiệu thiếu giống lúc dự đoán."""
    missing_columns = set(features) - set(frame.columns)
    if missing_columns:
        raise ValueError(f'Thiếu cột đầu vào: {sorted(missing_columns)}')
    result = frame.loc[:, features].replace(
        r'(?i)^\s*(?:na|n/a|nan|null|none|\?)?\s*$', np.nan, regex=True)
    for column in features:
        result[column] = pd.to_numeric(result[column], errors='raise')
    result = result.replace([np.inf, -np.inf], np.nan).astype(float)
    for column in set(features) - set(NUMERIC):
        present = result[column].dropna()
        if not (present % 1 == 0).all():
            raise ValueError(f'Cột {column} cần mã nguyên')
    for column in BINARY:
        if not result[column].dropna().isin([0, 1]).all():
            raise ValueError(f'Cột {column} cần giá trị 0/1')
    return result


def split_dataset(csv_path, output_dir, dataset):
    """Sắp theo thời gian, chia rồi xuất CSV; chưa học phép điền/mã hóa."""
    csv_path, output_dir = Path(csv_path), Path(output_dir)
    features, categorical = feature_columns(dataset)
    frame = pd.read_csv(csv_path, na_values=['?', 'null', 'None', 'none', 'NA', 'N/A'])
    required = set(features + ['instant', 'dteday', TARGET])
    if required - set(frame.columns):
        raise ValueError(f'{csv_path.name}: thiếu {sorted(required - set(frame.columns))}')
    original_rows = len(frame)
    original_missing = {c: int(v) for c, v in frame.isna().sum().items()}
    frame = frame.drop_duplicates().copy()
    duplicates = original_rows - len(frame)
    frame[TARGET] = pd.to_numeric(frame[TARGET], errors='raise').replace([np.inf, -np.inf], np.nan)
    missing_target = int(frame[TARGET].isna().sum())
    frame = frame.dropna(subset=[TARGET]).copy()
    if not ((frame[TARGET] >= 0) & (frame[TARGET] % 1 == 0)).all():
        raise ValueError('cnt phải là số nguyên không âm')
    frame[features] = normalize_features(frame, features)
    dates = pd.to_datetime(frame['dteday'], format='%Y-%m-%d', errors='raise')
    if dates.isna().any():
        raise ValueError('Thiếu ngày: không thể chia theo thời gian')
    if dataset == 'hour':
        if frame['hr'].isna().any() or not frame['hr'].between(0, 23).all():
            raise ValueError('Thiếu/sai giờ: không thể chia theo thời gian')
        dates = dates + pd.to_timedelta(frame['hr'], unit='h')
    frame['timestamp'] = dates
    if frame['timestamp'].duplicated().any() or frame['instant'].duplicated().any():
        raise ValueError('Trùng thời điểm hoặc mã bản ghi; cần kiểm tra dữ liệu')
    frame = frame.sort_values('timestamp', kind='stable').reset_index(drop=True)
    n = len(frame)
    a, b = n * 70 // 100, n * 70 // 100 + n * 15 // 100
    partitions = {'train': frame.iloc[:a].copy(),
                  'validation': frame.iloc[a:b].copy(), 'test': frame.iloc[b:].copy()}
    if any(len(part) < 2 for part in partitions.values()):
        raise ValueError('Mỗi tập cần ít nhất 2 dòng để đánh giá R²')
    if partitions['train'][features].isna().all().any():
        raise ValueError('Có cột hoàn toàn thiếu trên train, cần kiểm tra nguồn')
    raw_dir = output_dir / 'splits' / 'raw'
    raw_dir.mkdir(parents=True, exist_ok=True)
    report = {'dataset': dataset, 'target': TARGET, 'source_sha256': hashlib.sha256(csv_path.read_bytes()).hexdigest(),
              'original_rows': original_rows, 'missing_original': original_missing,
              'duplicates_removed': duplicates, 'missing_target_removed': missing_target,
              'features': features, 'categorical': categorical,
              'split_method': 'chronological: floor(70% N), floor(15% N), remainder',
              'splits': {}}
    manifest = []
    for name, part in partitions.items():
        part[features + [TARGET]].to_csv(raw_dir / f'{name}.csv', index=False)
        manifest.append(part[['instant', 'dteday', 'timestamp']].assign(split=name))
        report['splits'][name] = {'rows': len(part), 'percentage': 100 * len(part) / n,
                                 'start': str(part['timestamp'].iloc[0]),
                                 'end': str(part['timestamp'].iloc[-1])}
    pd.concat(manifest).to_csv(output_dir / 'splits' / 'split_manifest.csv', index=False)
    assert partitions['train']['timestamp'].max() < partitions['validation']['timestamp'].min()
    assert partitions['validation']['timestamp'].max() < partitions['test']['timestamp'].min()
    save_json(output_dir / 'split_report.json', report)
    return report


## Tiền xử lý và train, chọn siêu tham số chỉ bằng validation


In [6]:
def make_preprocessor(dataset):
    _, categorical = feature_columns(dataset)
    return ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), NUMERIC),
        ('binary', SimpleImputer(strategy='most_frequent'), BINARY),
        ('category', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical),
    ], remainder='drop', verbose_feature_names_out=False)


def regression_metrics(actual, predicted):
    return {'RMSE': float(np.sqrt(mean_squared_error(actual, predicted))),
            'MAE': float(mean_absolute_error(actual, predicted)),
            'R2': float(r2_score(actual, predicted))}


def train_and_select(train, validation, dataset, output_dir, param_grid=None, n_jobs=-1):
    """Không nhận test. Chọn RMSE validation thấp nhất, không refit trên 85%."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    features, _ = feature_columns(dataset)
    X_train, X_val = normalize_features(train, features), normalize_features(validation, features)
    y_train, y_val = train[TARGET], validation[TARGET]
    processor = make_preprocessor(dataset)
    train_encoded = processor.fit_transform(X_train)
    val_encoded = processor.transform(X_val)
    assert np.isfinite(train_encoded).all() and np.isfinite(val_encoded).all()
    candidates = list(ParameterGrid(DEFAULT_GRID if param_grid is None else param_grid))
    if not candidates:
        raise ValueError('Lưới tham số rỗng')
    history, best_model, best_score, best_params = [], None, np.inf, None
    for index, params in enumerate(candidates, start=1):
        model = RandomForestRegressor(random_state=SEED, n_jobs=n_jobs, **params)
        model.fit(train_encoded, y_train)
        metrics = regression_metrics(y_val, model.predict(val_encoded))
        history.append({'candidate': index, 'parameters': json.dumps(params), **metrics})
        print(f"{dataset}: cấu hình {index}/{len(candidates)}, validation RMSE={metrics['RMSE']:.3f}", flush=True)
        if metrics['RMSE'] < best_score:
            best_model, best_score, best_params = model, metrics['RMSE'], params
    # Hai thành phần đều đã fit; ghép pipeline không huấn luyện lại.
    pipeline = Pipeline([('preprocessor', processor), ('model', best_model)])
    table = pd.DataFrame(history).sort_values(['RMSE', 'candidate'], kind='stable')
    table.to_csv(output_dir / 'validation_results.csv', index=False)
    selection = {'algorithm': 'RandomForestRegressor', 'selection_metric': 'validation RMSE',
                 'best_params': best_params, 'best_validation_RMSE': float(best_score),
                 'fit_partition': 'train only (70%)', 'random_state': SEED,
                 'candidate_count': len(candidates),
                 'baseline_validation': regression_metrics(y_val, np.full(len(y_val), y_train.mean()))}
    save_json(output_dir / 'selection.json', selection)
    return pipeline, selection


## Đánh giá cuối, xuất dữ liệu đã mã hóa, lưu và tải lại mô hình


In [7]:
def export_encoded_splits(pipeline, output_dir, features):
    output_dir = Path(output_dir)
    processor = pipeline.named_steps['preprocessor']
    names = processor.get_feature_names_out()
    for split in ['train', 'validation', 'test']:
        raw = pd.read_csv(output_dir / 'splits' / 'raw' / f'{split}.csv')
        values = processor.transform(normalize_features(raw, features))
        if not np.isfinite(values).all():
            raise ValueError(f'{split}: dữ liệu mã hóa còn giá trị thiếu/vô hạn')
        encoded = pd.DataFrame(values, columns=names)
        encoded[TARGET] = raw[TARGET].to_numpy()
        encoded.to_csv(output_dir / 'splits' / f'{split}.csv', index=False)
    joblib.dump(processor, output_dir / 'preprocessor.joblib')


def evaluate_and_save(pipeline, selection, train, test, dataset, output_dir):
    """Đánh giá cấu hình đã chốt, lưu đúng mô hình được đánh giá."""
    output_dir = Path(output_dir)
    features, _ = feature_columns(dataset)
    X_test = normalize_features(test, features)
    predicted = pipeline.predict(X_test)
    test_metrics = regression_metrics(test[TARGET], predicted)
    baseline = regression_metrics(test[TARGET], np.full(len(test), train[TARGET].mean()))
    report = {'dataset': dataset, 'test': test_metrics, 'baseline_test': baseline,
              'baseline': 'mean cnt from train', 'selection': selection,
              'evaluated_at_utc': datetime.now(timezone.utc).isoformat()}
    save_json(output_dir / 'metrics.json', report)
    pd.DataFrame({'actual_cnt': test[TARGET].to_numpy(), 'predicted_cnt': predicted,
                  'residual': test[TARGET].to_numpy() - predicted}).to_csv(
                      output_dir / 'test_predictions.csv', index=False)
    versions = {'python': platform.python_version(), 'numpy': np.__version__,
                'pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__}
    bundle = {'pipeline': pipeline, 'features': features, 'target': TARGET,
              'dataset': dataset, 'versions': versions, 'report': report}
    path = output_dir / 'model.joblib'
    joblib.dump(bundle, path, compress=3)
    reloaded = joblib.load(path)
    np.testing.assert_allclose(reloaded['pipeline'].predict(X_test.iloc[:10]), predicted[:10])
    save_json(output_dir / 'environment.json', versions)
    (output_dir / 'requirements.txt').write_text('\n'.join(
        f'{name}=={version}' for name, version in versions.items() if name != 'python') + '\n', encoding='utf-8')
    return report


def predict_saved_model(model_path, new_data):
    """Đầu vào là DataFrame gốc hoặc đường dẫn CSV, không cần cnt."""
    bundle = joblib.load(model_path)
    frame = pd.read_csv(new_data) if isinstance(new_data, (str, Path)) else new_data.copy()
    X = normalize_features(frame, bundle['features'])
    return pd.DataFrame({'predicted_cnt': bundle['pipeline'].predict(X)}, index=frame.index)


## Hàm chạy trọn quy trình cho một dataset


In [8]:
def run_experiment(dataset, data_dir, run_dir, param_grid=None, n_jobs=-1):
    output_dir = Path(run_dir) / dataset
    if (output_dir / 'metrics.json').exists():
        raise FileExistsError('Lần chạy này đã có kết quả test. Tạo RUN_DIR mới để không ghi đè.')
    split_report = split_dataset(Path(data_dir) / f'{dataset}.csv', output_dir, dataset)
    raw_dir = output_dir / 'splits' / 'raw'
    train = pd.read_csv(raw_dir / 'train.csv')
    validation = pd.read_csv(raw_dir / 'validation.csv')
    pipeline, selection = train_and_select(train, validation, dataset, output_dir, param_grid, n_jobs)
    # Chỉ tải tập test vào bước đánh giá sau khi đã chọn xong cấu hình.
    test = pd.read_csv(raw_dir / 'test.csv')
    report = evaluate_and_save(pipeline, selection, train, test, dataset, output_dir)
    export_encoded_splits(pipeline, output_dir, split_report['features'])
    print(f'Đã lưu {dataset}: {output_dir}', flush=True)
    return report


## 1. Kết nối Drive và kiểm tra hai CSV gốc

Nếu dùng đường dẫn khác, sửa `PROJECT_NAME` hoặc `DATA_DIR`.
Không dùng `processed/.../train.csv` làm dữ liệu gốc vì file đó đã bị chia và mã hóa.


In [9]:
from IPython.display import display
from uuid import uuid4

PROJECT_NAME = 'TT_CSN'
PROJECT_DIR = connect_google_drive(PROJECT_NAME)
DATA_DIR = PROJECT_DIR / 'data'
DATASETS = ['day', 'hour']  # Có thể chọn ['day'] hoặc ['hour'].
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8]
RUN_DIR = PROJECT_DIR / 'runs' / RUN_ID

for dataset in DATASETS:
    path = DATA_DIR / f'{dataset}.csv'
    if not path.is_file():
        raise FileNotFoundError(f'Hãy tải CSV gốc lên: {path}')
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('Dữ liệu:', DATA_DIR)
print('Kết quả sẽ lưu:', RUN_DIR)


Mounted at /content/drive
Dữ liệu: /content/drive/MyDrive/TT_CSN/data
Kết quả sẽ lưu: /content/drive/MyDrive/TT_CSN/runs/20261004_014306_709f7fb3


## 2. Chia và xuất train / validation / test

Xuất bản chưa mã hóa tại `<dataset>/splits/raw/`. File CSV chứa các đặc trưng và cột nhãn `cnt`.
Ngày và mã bản ghi được lưu riêng trong `split_manifest.csv` để đối chiếu.
Không tự tạo nhãn thiếu; cột số điền trung vị, cột phân loại điền mode ở bước train.


In [10]:
split_reports = {}
split_summary = []
for dataset in DATASETS:
    report = split_dataset(DATA_DIR / f'{dataset}.csv', RUN_DIR / dataset, dataset)
    split_reports[dataset] = report
    for split, details in report['splits'].items():
        split_summary.append({'dataset': dataset, 'split': split, **details})
display(pd.DataFrame(split_summary))


,dataset,split,rows,percentage,start,end
0,day,train,511,69.904241,2011-01-01 00:00:00,2012-05-25 00:00:00
1,day,validation,109,14.911081,2012-05-26 00:00:00,2012-09-11 00:00:00
2,day,test,111,15.184679,2012-09-12 00:00:00,2012-12-31 00:00:00
3,hour,train,12165,69.998274,2011-01-01 00:00:00,2012-05-27 01:00:00
4,hour,validation,2606,14.995109,2012-05-27 02:00:00,2012-09-12 15:00:00
5,hour,test,2608,15.006617,2012-09-12 16:00:00,2012-12-31 23:00:00


## 3. Huấn luyện và chọn siêu tham số

Hàm `train_and_select()` chỉ nhận train và validation. Bộ điền thiếu và one-hot chỉ fit trên train.
Mã phân loại mới được xử lý bằng nhóm one-hot toàn 0 (`handle_unknown='ignore'`).
Giữ cột nhị phân 0/1; các biến thời tiết đã được chuẩn hóa trong nguồn, không scale lại.

Sửa lưới bên dưới **trước khi xem kết quả test**. RMSE validation càng thấp càng tốt.


In [11]:
PARAM_GRID = {
    'n_estimators': [100, 200],
    'max_depth': [None, 16],
    'min_samples_leaf': [1, 3],
    'max_features': [1.0],
}
selected = {}
for dataset in DATASETS:
    destination = RUN_DIR / dataset
    if (destination / 'metrics.json').exists():
        raise RuntimeError('Đã đánh giá test trong lần chạy này. Không chọn lại tham số dựa trên test.')
    train = pd.read_csv(destination / 'splits/raw/train.csv')
    validation = pd.read_csv(destination / 'splits/raw/validation.csv')
    pipeline, selection = train_and_select(train, validation, dataset, destination, PARAM_GRID)
    selected[dataset] = (pipeline, selection)
    print(dataset, 'tham số tốt nhất:', selection['best_params'])
    display(pd.read_csv(destination / 'validation_results.csv'))


day: cấu hình 1/8, validation RMSE=857.211
day: cấu hình 2/8, validation RMSE=839.359
day: cấu hình 3/8, validation RMSE=869.311
day: cấu hình 4/8, validation RMSE=842.272
day: cấu hình 5/8, validation RMSE=847.032
day: cấu hình 6/8, validation RMSE=834.433
day: cấu hình 7/8, validation RMSE=867.481
day: cấu hình 8/8, validation RMSE=842.535
day tham số tốt nhất: {'max_depth': 16, 'max_features': 1.0, 'min_samples_leaf': 1, 'n_estimators': 200}


,candidate,parameters,RMSE,MAE,R2
0,6,"{""max_depth"": 16, ""max_features"": 1.0, ""min_sa...",834.433083,663.033587,0.038539
1,2,"{""max_depth"": null, ""max_features"": 1.0, ""min_...",839.358603,677.560459,0.027155
2,4,"{""max_depth"": null, ""max_features"": 1.0, ""min_...",842.271506,679.373095,0.020391
3,8,"{""max_depth"": 16, ""max_features"": 1.0, ""min_sa...",842.534835,678.429560,0.019778
4,5,"{""max_depth"": 16, ""max_features"": 1.0, ""min_sa...",847.031816,675.869817,0.009286
5,1,"{""max_depth"": null, ""max_features"": 1.0, ""min_...",857.210516,700.676789,-0.014667
6,7,"{""max_depth"": 16, ""max_features"": 1.0, ""min_sa...",867.481023,711.630855,-0.039127
7,3,"{""max_depth"": null, ""max_features"": 1.0, ""min_...",869.311282,715.180901,-0.043517


hour: cấu hình 1/8, validation RMSE=126.380
hour: cấu hình 2/8, validation RMSE=126.936
hour: cấu hình 3/8, validation RMSE=144.145
hour: cấu hình 4/8, validation RMSE=146.272
hour: cấu hình 5/8, validation RMSE=128.348
hour: cấu hình 6/8, validation RMSE=129.473
hour: cấu hình 7/8, validation RMSE=144.974
hour: cấu hình 8/8, validation RMSE=147.594
hour tham số tốt nhất: {'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 1, 'n_estimators': 100}


,candidate,parameters,RMSE,MAE,R2
0,1,"{""max_depth"": null, ""max_features"": 1.0, ""min_...",126.379972,87.754007,0.679490
1,2,"{""max_depth"": null, ""max_features"": 1.0, ""min_...",126.935670,87.632736,0.676665
2,5,"{""max_depth"": 16, ""max_features"": 1.0, ""min_sa...",128.348437,91.392931,0.669428
3,6,"{""max_depth"": 16, ""max_features"": 1.0, ""min_sa...",129.472715,91.959992,0.663611
4,3,"{""max_depth"": null, ""max_features"": 1.0, ""min_...",144.145020,103.434248,0.583050
5,7,"{""max_depth"": 16, ""max_features"": 1.0, ""min_sa...",144.973626,105.010313,0.578242
6,4,"{""max_depth"": null, ""max_features"": 1.0, ""min_...",146.271675,104.542536,0.570656
7,8,"{""max_depth"": 16, ""max_features"": 1.0, ""min_sa...",147.594415,106.522113,0.562856


## 4. Đánh giá cuối và lưu mô hình lên Drive

Chỉ chạy sau khi đã chốt mô hình. Lưu đúng mô hình đã học trên 70% và được đánh giá trên test.
`model.joblib` chứa pipeline tiền xử lý + Random Forest, thứ tự cột và thông tin phiên bản.
File `splits/train.csv`, `splits/validation.csv`, `splits/test.csv` là bản số đã mã hóa,
có thể tách `X = df.drop(columns='cnt')`, `y = df['cnt']` để dùng với thuật toán khác.
Pipeline đã lưu nhận **dữ liệu gốc**, không nhận các CSV đã one-hot.


In [12]:
final_results = []
for dataset in DATASETS:
    destination = RUN_DIR / dataset
    if (destination / 'metrics.json').exists():
        raise RuntimeError(f'{dataset}: đã đánh giá test; đọc metrics.json để xem lại.')
    pipeline, selection = selected[dataset]
    train = pd.read_csv(destination / 'splits/raw/train.csv')
    test = pd.read_csv(destination / 'splits/raw/test.csv')
    report = evaluate_and_save(pipeline, selection, train, test, dataset, destination)
    export_encoded_splits(pipeline, destination, split_reports[dataset]['features'])
    final_results.append({'dataset': dataset, **report['test'],
                          'baseline_RMSE': report['baseline_test']['RMSE']})
summary = pd.DataFrame(final_results)
summary.to_csv(RUN_DIR / 'summary.csv', index=False)
display(summary)
print('Đã lưu mô hình, dữ liệu và báo cáo:', RUN_DIR)


,dataset,RMSE,MAE,R2,baseline_RMSE
0,day,1362.334476,1209.683739,0.537565,2679.246752
1,hour,90.164104,61.356375,0.824337,228.106134


Đã lưu mô hình, dữ liệu và báo cáo: /content/drive/MyDrive/TT_CSN/runs/20261004_014306_709f7fb3


## 5. Tải lại mô hình và dự đoán

Ví dụ dùng 5 dòng train chỉ để minh họa hàm dự đoán, không phải một phép đánh giá mới.
Thay `sample` bằng DataFrame mới hoặc đường dẫn CSV mới để sử dụng thực tế.
Trong phiên Colab mới: chạy ô thư viện, các ô định nghĩa hàm, kết nối Drive rồi gán
`RUN_DIR = PROJECT_DIR / 'runs' / '<mã lần chạy đã lưu>'`; không cần huấn luyện lại.
Khi tải lại mô hình, dùng phiên bản thư viện ghi trong `requirements.txt` của lần chạy.


In [13]:
DATASET_TO_PREDICT = DATASETS[0]
MODEL_PATH = RUN_DIR / DATASET_TO_PREDICT / 'model.joblib'
sample = pd.read_csv(RUN_DIR / DATASET_TO_PREDICT / 'splits/raw/train.csv').head(5).drop(columns=TARGET)
display(predict_saved_model(MODEL_PATH, sample))
# Dự đoán file mới:
# predictions = predict_saved_model(MODEL_PATH, PROJECT_DIR / 'data/new_data.csv')
# predictions.to_csv(PROJECT_DIR / 'predictions.csv', index=False)


,predicted_cnt
0,1217.170
1,1065.615
2,1395.110
3,1543.935
4,1590.835


## Các file được lưu

```
MyDrive/TT_CSN/runs/<RUN_ID>/
  summary.csv
  day/ (hour/ có cùng cấu trúc)
    splits/raw/train.csv, validation.csv, test.csv
    splits/train.csv, validation.csv, test.csv
    splits/split_manifest.csv
    split_report.json
    validation_results.csv
    selection.json
    metrics.json
    test_predictions.csv
    preprocessor.joblib
    model.joblib
    environment.json
    requirements.txt
```

Hàm chạy trọn quy trình nếu dùng Python thay vì các ô từng bước:
`run_experiment('day', DATA_DIR, RUN_DIR)` và tương tự với `'hour'`.
Không chạy lại hàm này trong cùng thư mục đã đánh giá test.

Tài liệu: [Kết nối Google Drive trong Colab](https://colab.research.google.com/notebooks/io.ipynb),
[RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html),
[OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html),
[Lưu mô hình](https://scikit-learn.org/stable/model_persistence.html).
Nguồn dữ liệu và thông tin trích dẫn: `Readme.txt` đi kèm dataset.
